In [1]:
import os
import re

import numpy as np
import scanpy as sc
import squidpy as sq
import anndata as ad

%load_ext autoreload
%autoreload 2

In [2]:
ad.__version__

'0.10.9'

## Load data

Download the Visium data from https://zenodo.org/records/6578047.
Download all files whose names start with "Visium" and place them in the ```processed``` folder within the user-specified working directory (```WD```).

In [3]:
WD = "/data/visium_heart"  # change as needed

In [4]:
input_dir = os.path.join(WD, "processed")

# Find all .h5ad files in the directory
h5ad_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(".h5ad")]

# Load each file into an AnnData object
adatas = [sc.read_h5ad(f) for f in h5ad_files]

# Extract clean sample names for each file using regex
sample_names = [re.search(r'Visium_(.+?)\.h5ad', os.path.basename(f)).group(1) for f in h5ad_files]

# Concatenate all AnnData objects
# Adjust `join='outer'` or `join='inner'` depending on whether you want to keep all or only common genes
adata_combined = ad.AnnData.concatenate(*adatas, join='inner', batch_key='sample', batch_categories=sample_names)
adata_combined

/tmp/ipykernel_3880255/1306160217.py:14: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_combined = ad.AnnData.concatenate(*adatas, join='inner', batch_key='sample', batch_categories=sample_names)


AnnData object with n_obs × n_vars = 88704 × 11681
    obs: 'n_counts', 'n_genes', 'percent.mt', 'Adipocyte', 'Cardiomyocyte', 'Endothelial', 'Fibroblast', 'Lymphoid', 'Mast', 'Myeloid', 'Neuronal', 'Pericyte', 'Cycling.cells', 'vSMCs', 'cell_type_original', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'sample'
    var: 'features'
    obsm: 'X_pca', 'X_spatial', 'X_umap'

## Inspect data

In [5]:
print(np.min(adata_combined.X))
print(np.max(adata_combined.X))

0.0
8.193758066884202


In [6]:
# Count in how many cells each gene is expressed
gene_expression_counts = np.array((adata_combined.X > 0).sum(axis=0)).flatten()
min_cells_per_gene = gene_expression_counts.min()
print(f"Minimum number of cells a gene is expressed in: {min_cells_per_gene}")

Minimum number of cells a gene is expressed in: 786


In [7]:
# Count how many genes are expressed in each cell
cell_expression_counts = np.array((adata_combined.X > 0).sum(axis=1)).flatten()
min_genes_per_cell = cell_expression_counts.min()
print(f"Minimum number of genes a cell expresses: {min_genes_per_cell}")

Minimum number of genes a cell expresses: 292


In [8]:
adata_combined.obs['sample'].value_counts()

sample
RZ_BZ_P3         4659
GT_IZ_P9         4361
control_P1       4269
FZ_GT_P4         4253
IZ_BZ_P2         4203
GT_IZ_P9_rep2    4113
IZ_P3            3771
IZ_P10           3646
RZ_P9            3626
GT_IZ_P15        3572
RZ_GT_P2         3538
RZ_P6            3484
RZ_BZ_P12        3392
RZ_BZ_P2         3373
FZ_P14           3175
FZ_GT_P19        3100
IZ_P15           3083
RZ_FZ_P5         3082
RZ_P3            2994
control_P7       2931
IZ_P16           2713
FZ_P18           2551
control_P8       2456
FZ_P20           2410
control_P17      2043
RZ_P11           2016
GT_IZ_P13        1890
Name: count, dtype: int64

## Continue building adata

In [9]:
assay_map = {
    'EFO:0010961': 'Visium Spatial Gene Expression'
}

cell_type_map = {
    'CL:0000513': 'cardiac muscle myoblast',
    'CL:0002548': 'fibroblast of cardiac tissue',
    'CL:0010008': 'cardiac endothelial cell',
    'CL:0001082': 'immature innate lymphoid cell',
    'CL:0000003': 'native cell',
    'CL:0000514': 'smooth muscle myoblast',
    'CL:0000669': 'pericyte cell',
    'CL:0000838': 'lymphoid lineage restricted progenitor cell',
    'CL:0000006': 'neuronal receptor cell', 
    'CL:0000097': 'mast cell',
    'CL:1000311': 'adipocyte of epicardial fat of left ventricle'
}

development_stage_map = {
    'HsapDv:0000138': '44-year-old human stage', 
    'HsapDv:0000151': '57-year-old human stage',
    'HsapDv:0000146': '52-year-old human stage',
    'HsapDv:0000137': '43-year-old human stage',
    'HsapDv:0000160': '66-year-old human stage',
    'HsapDv:0000168': '74-year-old human stage',
    'HsapDv:0000132': '38-year-old human stage',
    'HsapDv:0000141': '47-year-old human stage',
    'HsapDv:0000134': '40-year-old human stage',
    'HsapDv:0000152': '58-year-old human stage',
    'HsapDv:0000157': '63-year-old human stage',
    'HsapDv:0000149': '55-year-old human stage',
    'HsapDv:0000158': '64-year-old human stage',
    'HsapDv:0000155': '61-year-old human stage',
    'HsapDv:0000154': '60-year-old human stage',
    'HsapDv:0000145': '51-year-old human stage'
}

disease_map = {
    'MONDO:0005068': 'myocardial infarction',
    'PATO:0000461': 'normal'
}

ethnicity_map = {
    'HANCESTRO:0005': 'European'
}

organism_map = {
    'NCBITaxon:9606': 'Homo sapiens'
}

sex_map = {
    'PATO:0000383': 'female',
    'PATO:0000384': 'male'
}

tissue_map = {
    'UBERON:0002084': 'heart left ventricle'
}

In [10]:
adata_combined.obs['assay'] = adata_combined.obs['assay_ontology_term_id'].map(assay_map)
adata_combined.obs['cell_type'] = adata_combined.obs['cell_type_ontology_term_id'].map(cell_type_map)
adata_combined.obs['development_stage'] = adata_combined.obs['development_stage_ontology_term_id'].map(development_stage_map)
adata_combined.obs['disease'] = adata_combined.obs['disease_ontology_term_id'].map(disease_map)
adata_combined.obs['ethnicity'] = adata_combined.obs['ethnicity_ontology_term_id'].map(ethnicity_map)
adata_combined.obs['organism'] = adata_combined.obs['organism_ontology_term_id'].map(organism_map)
adata_combined.obs['sex'] = adata_combined.obs['sex_ontology_term_id'].map(sex_map)
adata_combined.obs['tissue'] = adata_combined.obs['tissue_ontology_term_id'].map(tissue_map)

In [11]:
adata_combined.obsm['spatial'] = adata_combined.obsm['X_spatial'].copy()

## Save adata

In [12]:
adata_combined.write_h5ad(os.path.join(WD, "adata_combined.h5ad"))